# Capa Silver — Limpieza y validación

Construcción de tablas Silver a partir de las tablas Bronze.

## 1. Setup — Lectura de tablas Bronze

In [0]:
# Importar funciones de PySpark y cargar las 4 tablas Bronze
from pyspark.sql import functions as F

df_pacientes_bronze = spark.table("workspace.bronze.pacientes")
df_citas_bronze = spark.table("workspace.bronze.citas")
df_eventos_bronze = spark.table("workspace.bronze.eventos_clinicos")
df_facturacion_bronze = spark.table("workspace.bronze.facturacion")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

## 2. Funciones reutilizables de calidad
Se usan en las 4 tablas para evitar repetir lógica.

In [0]:
# Normaliza nombres de columnas (minúsculas, sin espacios)
def normalizar_nombres_columnas(df):
    columnas_normalizadas = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df.toDF(*columnas_normalizadas)

# Cuenta nulos por columna
def contar_nulos(df):
    return df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

# Cuenta filas 100% duplicadas
def contar_duplicados_exactos(df):
    return df.count() - df.dropDuplicates().count()

# Busca valores repetidos en una columna clave (ej: id_paciente)
def revisar_clave_duplicada(df, columna_clave):
    return df.groupBy(columna_clave).count().filter(F.col("count") > 1).orderBy(F.desc("count"))

## 3. Limpieza por tabla

### 3.1 Pacientes

In [0]:
# Normalizar columnas y diagnosticar calidad
df_pacientes_silver = normalizar_nombres_columnas(df_pacientes_bronze)

display(contar_nulos(df_pacientes_silver))
print("Duplicados exactos:", contar_duplicados_exactos(df_pacientes_silver))
display(revisar_clave_duplicada(df_pacientes_silver, "id_paciente"))

In [0]:
# Convertir fecha_nacimiento a fecha real (registro corrupto -> queda como null)
df_pacientes_silver = df_pacientes_silver.withColumn(
    "fecha_nacimiento",
    F.try_to_date(F.col("fecha_nacimiento"), "yyyy-MM-dd HH:mm:ss")
)

nulos_fecha = df_pacientes_silver.filter(F.col("fecha_nacimiento").isNull()).count()
print(f"Pacientes con fecha_nacimiento inválida: {nulos_fecha}")

### 3.2 Citas

In [0]:
# Normalizar columnas y diagnosticar calidad
df_citas_silver = normalizar_nombres_columnas(df_citas_bronze)

display(contar_nulos(df_citas_silver))
print("Duplicados exactos:", contar_duplicados_exactos(df_citas_silver))
display(revisar_clave_duplicada(df_citas_silver, "id_cita"))

In [0]:
# Validar regla de negocio: motivo_cancelacion solo debería faltar si NO fue cancelada
citas_canceladas_sin_motivo = df_citas_silver.filter(
    (F.col("estado_cita") == "Cancelada") & F.col("motivo_cancelacion").isNull()
).count()
print(f"Citas canceladas sin motivo: {citas_canceladas_sin_motivo}")

### 3.3 Eventos clínicos

In [0]:
# Normalizar columnas y diagnosticar calidad
df_eventos_silver = normalizar_nombres_columnas(df_eventos_bronze)

display(contar_nulos(df_eventos_silver))
print("Duplicados exactos:", contar_duplicados_exactos(df_eventos_silver))
display(revisar_clave_duplicada(df_eventos_silver, "id_evento"))

### 3.4 Facturación

In [0]:
# Normalizar columnas y diagnosticar calidad
df_facturacion_silver = normalizar_nombres_columnas(df_facturacion_bronze)

display(contar_nulos(df_facturacion_silver))
print("Duplicados exactos:", contar_duplicados_exactos(df_facturacion_silver))
display(revisar_clave_duplicada(df_facturacion_silver, "id_factura"))

In [0]:
# Validar regla de negocio: bruto - descuento debería ser = neto
facturas_inconsistentes = df_facturacion_silver.filter(
    F.abs((F.col("valor_bruto") - F.col("valor_descuento")) - F.col("valor_neto")) > 0.01
).count()
print(f"Facturas con bruto - descuento /= neto: {facturas_inconsistentes}")

## 4. Integridad referencial entre tablas

In [0]:
# Verificar que no existan registros "huérfanos" (apuntando a un ID que no existe)
citas_huerfanas = df_citas_silver.join(
    df_pacientes_silver.select("id_paciente"), on="id_paciente", how="left_anti"
).count()

eventos_huerfanos = df_eventos_silver.join(
    df_citas_silver.select("id_cita"), on="id_cita", how="left_anti"
).count()

facturas_huerfanas = df_facturacion_silver.join(
    df_citas_silver.select("id_cita"), on="id_cita", how="left_anti"
).count()

print(f"Citas con paciente inexistente: {citas_huerfanas}")
print(f"Eventos con cita inexistente: {eventos_huerfanos}")
print(f"Facturas con cita inexistente: {facturas_huerfanas}")

## 5. Guardar tablas Silver

In [0]:
# Escribir las 4 tablas Silver ya limpias
df_pacientes_silver.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.pacientes")
df_citas_silver.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.citas")
df_eventos_silver.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.eventos_clinicos")
df_facturacion_silver.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.facturacion")

print("Capa Silver completa")

## 6. Verificación final

In [0]:
# Confirmar que las 4 tablas existen y tienen datos
spark.sql("SHOW TABLES IN workspace.silver").show(truncate=False)

for tabla in ["pacientes", "citas", "eventos_clinicos", "facturacion"]:
    total = spark.table(f"workspace.silver.{tabla}").count()
    print(f"{tabla}: {total} filas")